# 📖 Grok 长篇小说自动写作工具

将你的角色设定和故事大纲，自动扩展为 10 章、每章约 8000 字的长篇小说，并导出为 PDF。

**使用前请先完成：**
1. 安装依赖：`pip install -r requirements.txt`
2. 在下方填入你的 Grok API Key
3. 编辑 `user_inputs/` 目录下的角色设定、情节、风格文件（或直接在下方修改字符串）

---
## 步骤 0：配置 Grok API

In [ ]:
# ============================================================
# ✏️ 请在此填入你的 Grok API Key
# ============================================================
# 从 x.ai 获取: https://console.x.ai

GROK_API_KEY = "xai-..."  # <-- 替换为你的真实 Key

# 可选：修改 API 参数
GROK_BASE_URL = "https://api.x.ai/v1"
GROK_MODEL = "grok-3-mini"       # 或 grok-3, grok-2-latest
GROK_TEMPERATURE = 1.0
GROK_MAX_TOKENS = 16384

print(f"✅ API 配置就绪，模型: {GROK_MODEL}")

---
## 步骤 1：输入角色设定与故事内容

### 方式 A：从 txt 文件读取（推荐）
编辑 `user_inputs/` 目录下的文件后，运行下方单元格加载。

In [ ]:
# 从文件加载（推荐）
# 编辑完成后运行此单元格

def load_txt(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

CHARACTERS = load_txt('user_inputs/characters.txt')
PLOT = load_txt('user_inputs/plot.txt')
STYLE = load_txt('user_inputs/style.txt')

print(f"✅ 角色设定: {len(CHARACTERS)} 字")
print(f"✅ 故事大纲: {len(PLOT)} 字")
print(f"✅ 写作风格: {len(STYLE)} 字")

### 方式 B：直接在 Python 字符串中编辑
如果不想用文件，可以直接修改下方字符串。

In [ ]:
# 方式 B（可选）：直接在字符串中编辑
# 如果用方式 A（文件读取），跳过此单元格

CHARACTERS_STRING = """【姓名】林墨
【年龄】28岁
【职业】古城修复师
...（在此填写你的角色设定）...
"""

PLOT_STRING = """故事发生在一座1300年历史的江南古城...
...（在此填写你的故事大纲）...
"""

STYLE_STRING = """简洁、干净的中文...
...（在此填写你的写作风格）...
"""

# 取消以下注释以使用方式 B
# CHARACTERS = CHARACTERS_STRING
# PLOT = PLOT_STRING
# STYLE = STYLE_STRING
# print("✅ 已使用方式 B（字符串输入）")

---
## 步骤 2：初始化客户端与生成器

In [ ]:
# 导入模块
from grok_client import GrokClient
from story_generator import NovelGenerator
from pdf_exporter import export_novel_to_pdf
import config as cfg

# 配置 API 参数（覆盖 config.py 中的默认值）
cfg.API_BASE_URL = GROK_BASE_URL
cfg.API_MODEL = GROK_MODEL
cfg.API_TEMPERATURE = GROK_TEMPERATURE
cfg.API_MAX_TOKENS = GROK_MAX_TOKENS

# 定义进度回调
def progress_callback(msg):
    print(msg)

# 初始化客户端
client = GrokClient(
    api_key=GROK_API_KEY,
    base_url=cfg.API_BASE_URL,
    model=cfg.API_MODEL,
    temperature=cfg.API_TEMPERATURE,
    max_tokens=cfg.API_MAX_TOKENS,
    timeout=cfg.API_TIMEOUT,
    max_retries=3,
    on_progress=progress_callback,
)

# 初始化生成器
generator = NovelGenerator(
    grok_client=client,
    characters=CHARACTERS,
    plot=PLOT,
    style=STYLE,
    output_dir="outputs",
    save_intermediates=True,
)

print("✅ 初始化完成，准备开始写作！")

---
## 步骤 3：生成 10 章大纲

程序会根据角色设定和故事大纲，调用 Grok API 生成 10 个章节的大纲。

In [ ]:
chapter_outlines = generator.generate_chapter_outlines()

# 预览结果
print("\n" + "=" * 50)
print("📖 十章大纲预览")
print("=" * 50)
for ch in chapter_outlines:
    print(f"\n{'─' * 40}")
    print(f"第 {ch['chapter']} 章：{ch['title']}")
    print(f"{'─' * 40}")
    print(ch['outline'][:200] + "..." if len(ch['outline']) > 200 else ch['outline'])

---
## 步骤 4：每章拆分为 10 小段

对每一章的大纲，调用 Grok API 拆分为 10 小段的情节要点。
每段要点约 50–80 字。

In [ ]:
generator.generate_all_segments()

---
## 步骤 5：逐段写作全部章节

针对每个小段的情节要点，逐段生成正文。
每段约 800 字，10 段完成一章（约 8000 字）。

**⚠️ 注意：此步骤耗时最长（10 章 × 10 段 ≈ 100 次 API 调用）。**

In [ ]:
# 逐段写作
generator.write_all_chapters()

total_chars = len(generator.full_novel)
print(f"\n{'=' * 50}")
print(f"🎉 小说全文完成！")
print(f"   总字数：约 {total_chars} 字")
print(f"{'=' * 50}")

---
## 步骤 6：导出 PDF

将完整小说文本渲染为排版精美的 PDF 文件。

In [ ]:
pdf_path = export_novel_to_pdf(
    novel_text=generator.full_novel,
    chapter_outlines=chapter_outlines,
    output_path="outputs/novel.pdf",
    title="长篇小说",
    author="Grok 自动写作",
)

print(f"\n📄 PDF 已保存至: {pdf_path}")
print("✅ 全部流程完成！")

---
## 附录：中途断点续写

如果某一步因 API 错误中断，可以从此处恢复，而不需要重新开始。

In [ ]:
# 断点续写示例：如果前面已经生成了一些章节，可以从指定章节继续
# 修改 start_chapter 为需要继续的章节编号（从 0 开始）

# start_chapter = 3  # 从第 4 章开始
# for idx in range(start_chapter, len(generator.chapter_outlines)):
#     generator.write_chapter(idx)
# 
# generator.assemble_novel()
# print("✅ 续写完成")